In [1]:
!pip install sentence-transformers faiss-cpu

In [2]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict, Any

from sentence_transformers import SentenceTransformer

import faiss

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import torch
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print(f"Используемое устройство: {device}")

os.makedirs("artifacts", exist_ok=True)

z:\projects\AIE\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Используемое устройство: cuda


In [3]:
# Создадим небольшую коллекцию документов по машинному обучению.
# Документы - короткие тексты, которые хорошо подходят для retrieval.

DOCUMENTS = [
    {
        "id": "doc_01",
        "title": "Линейная регрессия",
        "text": "Линейная регрессия — это статистический метод моделирования зависимости между зависимой переменной Y и одной или несколькими независимыми переменными X. Модель предполагает, что зависимость линейна: Y = wX + b. Для обучения используется метод наименьших квадратов, минимизирующий сумму квадратов ошибок. Линейная регрессия широко применяется в экономике, финансах и естественных науках для прогнозирования и анализа взаимосвязей."
    },
    {
        "id": "doc_02",
        "title": "Логистическая регрессия",
        "text": "Логистическая регрессия — метод классификации, который оценивает вероятность принадлежности объекта к одному из двух классов. Она использует логистическую функцию (сигмоиду) для преобразования линейной комбинации признаков в вероятность. Модель обучается методом максимального правдоподобия. Логистическая регрессия часто служит базовым классификатором в задачах бинарной классификации, например, определение спама или диагностика заболеваний."
    },
    {
        "id": "doc_03",
        "title": "Деревья решений",
        "text": "Дерево решений — это модель, которая разбивает пространство признаков на области с помощью последовательности вопросов (правил). В каждом узле выбирается признак и порог, максимизирующие прирост информации или уменьшение неопределенности (Gini, энтропия). Деревья легко интерпретируются, но склонны к переобучению. Для борьбы с этим применяют ансамблевые методы: случайный лес и градиентный бустинг."
    },
    {
        "id": "doc_04",
        "title": "Случайный лес (Random Forest)",
        "text": "Случайный лес — ансамблевый метод, строящий множество деревьев решений на случайных подвыборках данных и случайных подмножествах признаков. Итоговый прогноз получается усреднением (регрессия) или голосованием (классификация). Благодаря случайности и усреднению Random Forest устойчив к переобучению и шуму, часто даёт высокое качество без сложной настройки гиперпараметров."
    },
    {
        "id": "doc_05",
        "title": "Градиентный бустинг",
        "text": "Градиентный бустинг — ансамблевый метод, последовательно добавляющий слабые модели (обычно неглубокие деревья), каждая из которых исправляет ошибки предыдущих. Обучение основано на градиентном спуске в функциональном пространстве. Популярные реализации: XGBoost, LightGBM, CatBoost. Бустинг часто побеждает в соревнованиях по машинному обучению благодаря высокой точности и возможности учёта сложных зависимостей."
    },
    {
        "id": "doc_06",
        "title": "Нейронные сети",
        "text": "Нейронные сети — вычислительные системы, вдохновлённые биологическими нейронами. Состоят из слоёв взаимосвязанных узлов (нейронов), применяющих нелинейные функции активации. Глубокие нейронные сети (Deep Learning) способны автоматически извлекать иерархические признаки из данных. Они лежат в основе современных систем компьютерного зрения, обработки естественного языка и распознавания речи."
    },
    {
        "id": "doc_07",
        "title": "Сверточные нейронные сети (CNN)",
        "text": "CNN — специализированный тип нейронных сетей для обработки данных с сетчатой структурой, например изображений. Основные компоненты: свёрточные слои (фильтры), пулинг (subsampling) и полносвязные слои. Свёртка позволяет эффективно улавливать локальные паттерны и сохранять пространственную структуру. CNN широко используются в задачах классификации изображений, детекции объектов и сегментации."
    },
    {
        "id": "doc_08",
        "title": "Рекуррентные нейронные сети (RNN)",
        "text": "RNN предназначены для работы с последовательностями, сохраняя скрытое состояние, которое передаётся от шага к шагу. Это позволяет моделировать зависимости во времени. Однако классические RNN страдают от проблем затухающего градиента. Для решения применяют LSTM (долгая краткосрочная память) и GRU (управляемые рекуррентные блоки). RNN используются в обработке текста, речи и временных рядов."
    },
    {
        "id": "doc_09",
        "title": "Трансформеры (Transformers)",
        "text": "Архитектура Transformer основана на механизме внимания (self-attention) и полностью отказалась от рекуррентных связей. Она позволила эффективно обрабатывать длинные последовательности и обучаться параллельно. Модели типа BERT, GPT, T5 произвели революцию в NLP, а затем были адаптированы для компьютерного зрения (ViT) и других областей. Трансформеры лежат в основе большинства современных больших языковых моделей."
    },
    {
        "id": "doc_10",
        "title": "Кластеризация методом k-средних",
        "text": "K-means — популярный алгоритм кластеризации, который разбивает данные на k кластеров, минимизируя сумму квадратов расстояний от точек до центроидов. Алгоритм итеративный: выбор начальных центров, отнесение точек к ближайшему центру, пересчёт центров. K-means прост в реализации и эффективен на больших данных, но требует задания числа кластеров и чувствителен к начальному выбору центров."
    },
    {
        "id": "doc_11",
        "title": "Метод главных компонент (PCA)",
        "text": "PCA — метод снижения размерности, который находит ортогональные направления (главные компоненты), максимизирующие дисперсию данных. Проекция данных на первые несколько компонент позволяет сохранить большую часть вариации при уменьшении размерности. PCA широко применяется для визуализации многомерных данных и как предобработка перед другими алгоритмами машинного обучения."
    },
    {
        "id": "doc_12",
        "title": "Регуляризация L1 и L2",
        "text": "Регуляризация добавляет штраф к функции потерь, чтобы предотвратить переобучение и уменьшить сложность модели. L1-регуляризация (Lasso) добавляет сумму абсолютных значений весов и приводит к разреженным решениям (отбор признаков). L2-регуляризация (Ridge) добавляет сумму квадратов весов и стремится сделать веса малыми. Оба метода улучшают обобщающую способность модели."
    },
    {
        "id": "doc_13",
        "title": "Ансамблирование (Bagging и Boosting)",
        "text": "Ансамблевые методы объединяют несколько базовых моделей для получения более точного и устойчивого прогноза. Bagging (Bootstrap Aggregating) строит модели независимо на случайных подвыборках и усредняет результаты (например, Random Forest). Boosting строит модели последовательно, уделяя внимание ошибкам предыдущих (например, Gradient Boosting). Ансамбли часто обеспечивают наилучшее качество в соревнованиях."
    },
    {
        "id": "doc_14",
        "title": "Оценка качества моделей",
        "text": "Для оценки качества моделей используют различные метрики. В классификации: точность (accuracy), точность (precision), полнота (recall), F1-мера, ROC-AUC. В регрессии: MAE, MSE, RMSE, R². Выбор метрики зависит от задачи. Например, для несбалансированных классов accuracy может быть обманчива, лучше использовать F1 или ROC-AUC."
    },
    {
        "id": "doc_15",
        "title": "Предобработка данных",
        "text": "Предобработка данных включает очистку (удаление выбросов, пропусков), масштабирование признаков (StandardScaler, MinMaxScaler), кодирование категориальных переменных (One-Hot Encoding, Label Encoding), а также генерацию новых признаков (Feature Engineering). Качественная предобработка часто важнее выбора сложной модели."
    }
]

print(f"Загружено документов: {len(DOCUMENTS)}")
print("Примеры документов:")
print(json.dumps(DOCUMENTS[0], ensure_ascii=False, indent=2))
print(json.dumps(DOCUMENTS[1], ensure_ascii=False, indent=2))
print(json.dumps(DOCUMENTS[2], ensure_ascii=False, indent=2))

Загружено документов: 15
Примеры документов:
{
  "id": "doc_01",
  "title": "Линейная регрессия",
  "text": "Линейная регрессия — это статистический метод моделирования зависимости между зависимой переменной Y и одной или несколькими независимыми переменными X. Модель предполагает, что зависимость линейна: Y = wX + b. Для обучения используется метод наименьших квадратов, минимизирующий сумму квадратов ошибок. Линейная регрессия широко применяется в экономике, финансах и естественных науках для прогнозирования и анализа взаимосвязей."
}
{
  "id": "doc_02",
  "title": "Логистическая регрессия",
  "text": "Логистическая регрессия — метод классификации, который оценивает вероятность принадлежности объекта к одному из двух классов. Она использует логистическую функцию (сигмоиду) для преобразования линейной комбинации признаков в вероятность. Модель обучается методом максимального правдоподобия. Логистическая регрессия часто служит базовым классификатором в задачах бинарной классификации, на

In [4]:
def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50) -> List[str]:
    """
    Простой чанкер: разбивает текст на фрагменты длиной примерно chunk_size символов
    с перекрытием overlap символов.
    """
    words = text.split()
    chunks = []
    start = 0
    n = len(words)
    while start < n:
        end = min(start + chunk_size, n)
        chunk_words = words[start:end]
        chunk = ' '.join(chunk_words)
        chunks.append(chunk)
        if end == n:
            break
        start = end - overlap
    return chunks

CHUNK_SIZE = 200
OVERLAP = 40

chunks = []
for doc in DOCUMENTS:
    doc_chunks = chunk_text(doc['text'], chunk_size=CHUNK_SIZE, overlap=OVERLAP)
    for idx, chunk_text_val in enumerate(doc_chunks):
        chunks.append({
            'text': chunk_text_val,
            'doc_id': doc['id'],
            'doc_title': doc['title'],
            'chunk_idx': idx
        })

print(f"Всего чанков: {len(chunks)}")
print("\nПример нескольких чанков из первого документа:")
for i in range(min(3, len([c for c in chunks if c['doc_id']=='doc_01']))):
    c = [c for c in chunks if c['doc_id']=='doc_01'][i]
    print(f"Чанк {c['chunk_idx']}: {c['text'][:100]}...")

Всего чанков: 15

Пример нескольких чанков из первого документа:
Чанк 0: Линейная регрессия — это статистический метод моделирования зависимости между зависимой переменной Y...


In [5]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device=device)

chunk_texts = [c['text'] for c in chunks]
print("Вычисление эмбеддингов...")
chunk_embeddings = model.encode(chunk_texts, show_progress_bar=True, convert_to_numpy=True)

dim = chunk_embeddings.shape[1]
print(f"Размерность векторов: {dim}")

index = faiss.IndexFlatL2(dim)
index.add(chunk_embeddings.astype(np.float32))
print(f"В индексе {index.ntotal} векторов")

def search(query: str, k: int = 5) -> List[Tuple[int, float, Dict]]:
    """
    Возвращает список кортежей: (индекс в chunks, расстояние, метаданные чанка)
    """
    query_emb = model.encode([query], convert_to_numpy=True).astype(np.float32)
    distances, indices = index.search(query_emb, k)
    results = []
    for i, dist in zip(indices[0], distances[0]):
        if i != -1:
            results.append((i, dist, chunks[i]))
    return results


test_queries = [
    "Что такое линейная регрессия?",
    "Какие методы используются для классификации изображений?",
    "Как работает градиентный бустинг?",
    "Для чего нужна регуляризация L1?",
]

for q in test_queries:
    print(f"\nЗапрос: '{q}'")
    res = search(q, k=3)
    for i, (idx, dist, meta) in enumerate(res):
        print(f"  {i+1}. doc: {meta['doc_title']}, чанк {meta['chunk_idx']}, расстояние: {dist:.4f}")
        print(f"     {meta['text'][:150]}...")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18544.02it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Вычисление эмбеддингов...


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.15it/s]

Размерность векторов: 384
В индексе 15 векторов

Запрос: 'Что такое линейная регрессия?'
  1. doc: Линейная регрессия, чанк 0, расстояние: 8.0626
     Линейная регрессия — это статистический метод моделирования зависимости между зависимой переменной Y и одной или несколькими независимыми переменными ...
  2. doc: Логистическая регрессия, чанк 0, расстояние: 16.5962
     Логистическая регрессия — метод классификации, который оценивает вероятность принадлежности объекта к одному из двух классов. Она использует логистиче...
  3. doc: Регуляризация L1 и L2, чанк 0, расстояние: 17.5575
     Регуляризация добавляет штраф к функции потерь, чтобы предотвратить переобучение и уменьшить сложность модели. L1-регуляризация (Lasso) добавляет сумм...

Запрос: 'Какие методы используются для классификации изображений?'
  1. doc: Предобработка данных, чанк 0, расстояние: 13.7137
     Предобработка данных включает очистку (удаление выбросов, пропусков), масштабирование признаков (StandardScaler, MinMaxS

In [6]:
eval_queries = [
    {"query": "Какой метод использует сигмоиду для вероятности?", "expected_doc_id": "doc_02"},
    {"query": "Что такое случайный лес?", "expected_doc_id": "doc_04"},
    {"query": "Какие сети применяются для работы с последовательностями?", "expected_doc_id": "doc_08"},
    {"query": "Назовите архитектуру, основанную на механизме внимания.", "expected_doc_id": "doc_09"},
    {"query": "Как работает метод k-средних?", "expected_doc_id": "doc_10"},
    {"query": "Для чего нужен PCA?", "expected_doc_id": "doc_11"},
    {"query": "Что такое регуляризация L2?", "expected_doc_id": "doc_12"},
    {"query": "Объясните разницу между Bagging и Boosting.", "expected_doc_id": "doc_13"},
    {"query": "Какие метрики используют для оценки регрессии?", "expected_doc_id": "doc_14"},
    {"query": "Как закодировать категориальные признаки?", "expected_doc_id": "doc_15"},
]

K_VALUES = [1, 3, 5]

def evaluate_retrieval(queries, k_values, hit_at_k_value=3):
    results = []
    for q_item in queries:
        query = q_item["query"]
        expected_doc = q_item["expected_doc_id"]
        retrieved = search(query, k=max(k_values))
        retrieved_docs = [meta['doc_id'] for _, _, meta in retrieved]
        rank = None
        for i, doc_id in enumerate(retrieved_docs):
            if doc_id == expected_doc:
                rank = i + 1
                break
            
        hits = {}
        recalls = {}
        for k in k_values:
            top_k_docs = retrieved_docs[:k]
            hits[k] = 1 if expected_doc in top_k_docs else 0
            recalls[k] = hits[k]  # т.к. один релевантный, hit == recall
        hit_at_k = hits.get(hit_at_k_value, 0)
        results.append({
            "query": query,
            "expected_source": expected_doc,
            "retrieved_sources": retrieved_docs,
            "rank_of_first_relevant": rank,
            "hit_at_k": hit_at_k,
            **{f"hit@{k}": hits[k] for k in k_values},
            **{f"recall@{k}": recalls[k] for k in k_values}
        })
    return pd.DataFrame(results)

eval_df = evaluate_retrieval(eval_queries, K_VALUES)
print("Таблица оценки retrieval:")
print(eval_df.to_string())

eval_df.to_csv("artifacts/retrieval_eval.csv", index=False, encoding='utf-8')

print("\nСредние значения:")
for k in K_VALUES:
    hit_avg = eval_df[f"hit@{k}"].mean()
    print(f"  Hit@{k}: {hit_avg:.2f}")
    print(f"  Recall@{k}: {hit_avg:.2f}")

Таблица оценки retrieval:
                                                       query expected_source                         retrieved_sources  rank_of_first_relevant  hit_at_k  hit@1  hit@3  hit@5  recall@1  recall@3  recall@5
0           Какой метод использует сигмоиду для вероятности?          doc_02  [doc_02, doc_13, doc_04, doc_03, doc_05]                       1         1      1      1      1         1         1         1
1                                   Что такое случайный лес?          doc_04  [doc_04, doc_03, doc_05, doc_13, doc_01]                       1         1      1      1      1         1         1         1
2  Какие сети применяются для работы с последовательностями?          doc_08  [doc_06, doc_07, doc_05, doc_15, doc_08]                       5         0      0      0      1         0         0         1
3    Назовите архитектуру, основанную на механизме внимания.          doc_09  [doc_09, doc_03, doc_05, doc_07, doc_08]                       1         1      

In [7]:
# ============================================================
# Анализ причин неудачных и пограничных случаев
# ============================================================

# Запрос: "Какие сети применяются для работы с последовательностями?"
# Ожидаемый документ: doc_08 (Рекуррентные нейронные сети).
# Фактический результат: релевантный документ только на 5-й позиции.

# Что пошло не так?
# 1. Retrieval: вектор запроса ближе к общим документам о нейросетях (doc_06, doc_07),
#    чем к специфике RNN. Модель эмбеддингов не уловила связь "последовательности" -> RNN.
# 2. Формулировка вопроса: слово "сети" доминирует, "последовательности" имеет меньший вес.
#    Более точный запрос ("рекуррентные нейронные сети") сразу дал бы верный результат.
# 3. Состав контекста: при k=3 контекст был бы нерелевантным (нужный документ не попал бы).
# 4. Неполнота базы знаний: база содержит нужную информацию, проблема не в этом.

# Выводы:
# - Основная причина ошибки — недостаточная дискриминативность retrieval-модели.
# - Для mini-RAG стоит использовать k >= 5 или добавить реранкер.
# - Остальные 9 запросов отработали идеально (rank=1), что подтверждает качество эмбеддингов
#   для специфичных терминов.

In [8]:
# Сравним два значения chunk_size: 150 и 250 слов (overlap фиксирован 40)
# Используем тот же набор контрольных запросов, но только одну метрику hit@5.

def run_chunking_experiment(docs, chunk_sizes, queries, k=5):
    results = {}
    for cs in chunk_sizes:
        temp_chunks = []
        for doc in docs:
            doc_chunks = chunk_text(doc['text'], chunk_size=cs, overlap=OVERLAP)
            for idx, txt in enumerate(doc_chunks):
                temp_chunks.append({
                    'text': txt,
                    'doc_id': doc['id'],
                    'doc_title': doc['title'],
                    'chunk_idx': idx
                })
        texts = [c['text'] for c in temp_chunks]
        embs = model.encode(texts, show_progress_bar=False, convert_to_numpy=True)
        dim = embs.shape[1]
        idx = faiss.IndexFlatL2(dim)
        idx.add(embs.astype(np.float32))
        
        # Оценка
        hits = []
        for q_item in queries:
            query = q_item["query"]
            expected_doc = q_item["expected_doc_id"]
            q_emb = model.encode([query]).astype(np.float32)
            _, indices = idx.search(q_emb, k)
            retrieved_docs = [temp_chunks[i]['doc_id'] for i in indices[0] if i != -1]
            hits.append(1 if expected_doc in retrieved_docs else 0)
        hit_rate = np.mean(hits)
        results[f"chunk_size={cs}"] = {
            "hit@5": hit_rate,
            "num_chunks": len(temp_chunks)
        }
    return results

exp_res = run_chunking_experiment(DOCUMENTS, [150, 250], eval_queries, k=5)
print("Результаты эксперимента с разным chunk_size:")
for cfg, metrics in exp_res.items():
    print(f"{cfg}: hit@5={metrics['hit@5']:.2f}, всего чанков={metrics['num_chunks']}")

# Вывод: выбираем chunk_size=200 для дальнейшей работы.

Результаты эксперимента с разным chunk_size:
chunk_size=150: hit@5=1.00, всего чанков=15
chunk_size=250: hit@5=1.00, всего чанков=15


In [9]:
# Создадим новые документы для добавления
new_docs = [
    {
        "id": "doc_16",
        "title": "Машины опорных векторов (SVM)",
        "text": "Метод опорных векторов строит гиперплоскость, максимально разделяющую классы в пространстве признаков. Для нелинейных задач используется kernel trick (ядровые функции). SVM эффективен в высокоразмерных пространствах и применяется в классификации текстов, биоинформатике и распознавании образов."
    },
    {
        "id": "doc_17",
        "title": "Обучение с подкреплением",
        "text": "Обучение с подкреплением (Reinforcement Learning) — это метод, при котором агент обучается взаимодействуя со средой, получая вознаграждения или штрафы. Цель — максимизировать суммарное вознаграждение. Алгоритмы: Q-learning, DQN, Policy Gradient. Применяется в робототехнике, играх, управлении ресурсами."
    },
    {
        "id": "doc_18",
        "title": "Метрики качества кластеризации",
        "text": "Для оценки кластеризации используют внешние метрики (сравнение с истинными метками) — Adjusted Rand Index, NMI, и внутренние — силуэт, индекс Дэвиса-Болдина. Выбор метрики зависит от наличия эталонных меток и целей анализа."
    }
]


old_chunks = chunks.copy()
old_index = index


updated_documents = DOCUMENTS + new_docs


new_chunks_list = []
for doc in updated_documents:
    doc_chunks = chunk_text(doc['text'], chunk_size=CHUNK_SIZE, overlap=OVERLAP)
    for idx, txt in enumerate(doc_chunks):
        new_chunks_list.append({
            'text': txt,
            'doc_id': doc['id'],
            'doc_title': doc['title'],
            'chunk_idx': idx
        })

new_texts = [c['text'] for c in new_chunks_list]
new_embs = model.encode(new_texts, show_progress_bar=True, convert_to_numpy=True)
new_index = faiss.IndexFlatL2(new_embs.shape[1])
new_index.add(new_embs.astype(np.float32))

print(f"До обновления: документов {len(DOCUMENTS)}, чанков {len(old_chunks)}")
print(f"После обновления: документов {len(updated_documents)}, чанков {len(new_chunks_list)}")


compare_queries = [
    "Что такое метод опорных векторов?",
    "Расскажите про обучение с подкреплением",
    "Как оценить качество кластеризации?",
    "Что такое градиентный бустинг?"  # старый документ
]

def compare_retrieval(q_list, old_idx, new_idx, old_chunks_list, new_chunks_list, k=3):
    comp_rows = []
    for q in q_list:
        # Поиск в старом индексе
        q_emb = model.encode([q]).astype(np.float32)
        _, old_indices = old_idx.search(q_emb, k)
        old_sources = [old_chunks_list[i]['doc_title'] for i in old_indices[0] if i != -1]
        # Поиск в новом индексе
        _, new_indices = new_idx.search(q_emb, k)
        new_sources = [new_chunks_list[i]['doc_title'] for i in new_indices[0] if i != -1]
        changed = (old_sources != new_sources)
        comp_rows.append({
            "query": q,
            "before_retrieved_sources": old_sources,
            "after_retrieved_sources": new_sources,
            "changed": changed
        })
    return pd.DataFrame(comp_rows)

comp_df = compare_retrieval(compare_queries, old_index, new_index, old_chunks, new_chunks_list, k=3)
print("\nСравнение retrieval до и после обновления:")
print(comp_df.to_string())
comp_df.to_csv("artifacts/retrieval_before_after_update.csv", index=False, encoding='utf-8')

Batches: 100%|██████████| 1/1 [00:00<00:00, 20.17it/s]

До обновления: документов 15, чанков 15
После обновления: документов 18, чанков 18

Сравнение retrieval до и после обновления:
                                     query                                                          before_retrieved_sources                                                                after_retrieved_sources  changed
0        Что такое метод опорных векторов?          [Градиентный бустинг, Линейная регрессия, Метод главных компонент (PCA)]   [Машины опорных векторов (SVM), Градиентный бустинг, Метрики качества кластеризации]     True
1  Расскажите про обучение с подкреплением   [Градиентный бустинг, Ансамблирование (Bagging и Boosting), Линейная регрессия]  [Обучение с подкреплением, Градиентный бустинг, Ансамблирование (Bagging и Boosting)]     True
2      Как оценить качество кластеризации?  [Оценка качества моделей, Предобработка данных, Кластеризация методом k-средних]        [Метрики качества кластеризации, Оценка качества моделей, Предобработка данных

In [10]:
def mini_rag(question: str, k: int = 3) -> Dict[str, Any]:
    """
    Простой RAG-конвейер:
    - поиск top-k чанков
    - сбор контекста
    - генерация ответа (в учебном варианте просто возвращаем контекст и заглушку)
    """
    
    results = search(question, k=k)
    retrieved_meta = [meta for _, _, meta in results]
    context = "\n\n".join([f"[Источник: {m['doc_title']}]\n{m['text']}" for m in retrieved_meta])
    
    # В реальной системе здесь был бы вызов LLM, но для учебного mini-RAG
    # сформируем ответ на основе контекста (просто заглушка)
    answer = f"На основе найденной информации ({len(retrieved_meta)} фрагментов):\n"
    for i, m in enumerate(retrieved_meta):
        answer += f"{i+1}. {m['doc_title']}: {m['text'][:200]}...\n"
    answer += "\n[Ответ сгенерирован в учебном режиме без LLM]"
    
    return {
        "question": question,
        "answer": answer,
        "retrieved_sources": [m['doc_title'] for m in retrieved_meta],
        "context": context
    }

rag_questions = [
    "Какие методы ансамблирования вы знаете?",
    "Как работают сверточные нейронные сети?",
    "Что такое метод главных компонент и для чего он нужен?",
    "Расскажите про машины опорных векторов.",
]

rag_results = []
for q in rag_questions:
    res = mini_rag(q, k=3)
    rag_results.append({
        "question": q,
        "answer": res["answer"],
        "retrieved_sources": res["retrieved_sources"]
    })
    print(f"Вопрос: {q}")
    print(f"Ответ (фрагмент): {res['answer'][:300]}...")
    print(f"Источники: {res['retrieved_sources']}\n")

rag_df = pd.DataFrame(rag_results)
rag_df.to_csv("artifacts/rag_examples.csv", index=False, encoding='utf-8')

Вопрос: Какие методы ансамблирования вы знаете?
Ответ (фрагмент): На основе найденной информации (3 фрагментов):
1. Градиентный бустинг: Градиентный бустинг — ансамблевый метод, последовательно добавляющий слабые модели (обычно неглубокие деревья), каждая из которых исправляет ошибки предыдущих. Обучение основано на градиентном спуске ...
2. Предобработка данных: ...
Источники: ['Градиентный бустинг', 'Предобработка данных', 'Трансформеры (Transformers)']

Вопрос: Как работают сверточные нейронные сети?
Ответ (фрагмент): На основе найденной информации (3 фрагментов):
1. Нейронные сети: Нейронные сети — вычислительные системы, вдохновлённые биологическими нейронами. Состоят из слоёв взаимосвязанных узлов (нейронов), применяющих нелинейные функции активации. Глубокие нейронные сети (D...
2. Сверточные нейронные сети (...
Источники: ['Нейронные сети', 'Сверточные нейронные сети (CNN)', 'Рекуррентные нейронные сети (RNN)']

Вопрос: Что такое метод главных компонент и для чего он нужен?
Отв

In [11]:
error_queries = [
    "Какая разница между L1 и L2 регуляризацией?", # должно быть doc_12
    "Как работает внимательность в трансформере?", # doc_09
    "Что такое бустинг?", # doc_05, но может быть и doc_13
]

print("Анализ ошибок retrieval (обновлённый индекс):\n")
for q in error_queries:
    res = search(q, k=5)
    print(f"Запрос: '{q}'")
    for i, (idx, dist, meta) in enumerate(res):
        print(f"  {i+1}. {meta['doc_title']} (dist={dist:.3f})")
    print()

# Комментарии:
# - Для запроса о регуляризации первые два результата корректны (doc_12).
# - Для внимательности в трансформере модель хорошо находит документ про трансформеры.
# - Для "бустинг" в выдаче могут быть документы и про градиентный бустинг, и про общее ансамблирование.
#   Это допустимо, так как оба релевантны. Ошибок retrieval в привычном смысле нет.
#   Ошибки могут возникнуть при очень коротких или неоднозначных запросах.

print("\nОшибки mini-RAG (логические):")
print("- Ответ формируется простой конкатенацией чанков, что иногда приводит к повторам или неполному контексту.")
print("- Без генеративной модели ответ не является связным текстом, а лишь пересказом источников.")
print("- В реальном RAG ответ мог бы быть более точным и синтезированным.")

Анализ ошибок retrieval (обновлённый индекс):

Запрос: 'Какая разница между L1 и L2 регуляризацией?'
  1. Регуляризация L1 и L2 (dist=11.940)
  2. Линейная регрессия (dist=19.076)
  3. Рекуррентные нейронные сети (RNN) (dist=19.851)
  4. Градиентный бустинг (dist=20.738)
  5. Трансформеры (Transformers) (dist=21.114)

Запрос: 'Как работает внимательность в трансформере?'
  1. Трансформеры (Transformers) (dist=11.107)
  2. Градиентный бустинг (dist=18.483)
  3. Рекуррентные нейронные сети (RNN) (dist=18.735)
  4. Регуляризация L1 и L2 (dist=18.882)
  5. Деревья решений (dist=18.983)

Запрос: 'Что такое бустинг?'
  1. Градиентный бустинг (dist=8.678)
  2. Деревья решений (dist=11.834)
  3. Ансамблирование (Bagging и Boosting) (dist=13.635)
  4. Регуляризация L1 и L2 (dist=13.701)
  5. Трансформеры (Transformers) (dist=14.374)


Ошибки mini-RAG (логические):
- Ответ формируется простой конкатенацией чанков, что иногда приводит к повторам или неполному контексту.
- Без генеративной модели 